# Day 9 – Processed E-commerce Dataset

 loads the Orders, Customers, and Products CSV files, demonstrates `concat()` and `merge()`, uses `apply()` to create useful columns, performs DateTime operations, and exports the final processed e-commerce dataset as CSV.

In [2]:
import pandas as pd

orders = pd.read_csv('Day9_Orders.csv')
customers = pd.read_csv('Day9_Customers.csv')
products = pd.read_csv('Day9_Products.csv')

print('Orders:', orders.shape)
print('Customers:', customers.shape)
print('Products:', products.shape)

Orders: (120, 7)
Customers: (30, 5)
Products: (20, 5)


In [3]:
display(orders.head())
display(customers.head())
display(products.head())

,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered


,Customer_ID,Customer_Name,City,Region,Membership_Type
0,C001,Aarav Sharma,Srinagar,North,Premium
1,C002,Zoya Khan,Delhi,North,Regular
2,C003,Rohan Mehta,Mumbai,West,Premium
3,C004,Ananya Singh,Jammu,North,Regular
4,C005,Kabir Ali,Lucknow,North,New


,Product_ID,Product_Name,Category,Unit_Price,Brand
0,P001,Wireless Headphones,Electronics,1499,SoundMax
1,P002,Mechanical Keyboard,Electronics,2499,KeyPro
2,P003,Wireless Mouse,Electronics,899,TechGear
3,P004,Smart Watch,Electronics,3299,FitTech
4,P005,Power Bank,Electronics,1199,VoltPlus


## 1. Demonstrating `concat()`

The Orders DataFrame is split into two parts and then recombined vertically using `pd.concat()`. This demonstrates how DataFrames with the same structure can be combined.

In [4]:
orders_part1 = orders.iloc[:len(orders)//2].copy()
orders_part2 = orders.iloc[len(orders)//2:].copy()

orders_recombined = pd.concat(
    [orders_part1, orders_part2],
    ignore_index=True
)

print('Original Orders shape:', orders.shape)
print('Recombined Orders shape:', orders_recombined.shape)
display(orders_recombined.head())

Original Orders shape: (120, 7)
Recombined Orders shape: (120, 7)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered


## 2. Combining related information using `merge()`

Orders is merged with Customers using `Customer_ID`, and then with Products using `Product_ID`. A left join keeps every order while adding the matching customer and product details.

In [5]:
processed = orders_recombined.merge(
    customers,
    on='Customer_ID',
    how='left'
).merge(
    products,
    on='Product_ID',
    how='left'
)

display(processed.head())
print('Merged shape:', processed.shape)

,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status,Customer_Name,City,Region,Membership_Type,Product_Name,Category,Unit_Price,Brand
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered,Harsh Vardhan,Noida,North,Premium,Cricket Bat,Sports,2499,BatPro
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered,Ishita Gupta,Bengaluru,South,Premium,Wireless Mouse,Electronics,899,TechGear
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered,Karan Joshi,Chandigarh,North,Regular,Smart Watch,Electronics,3299,FitTech
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered,Maryam Khan,Hyderabad,South,Regular,Machine Learning Basics,Books,999,AIPress
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered,Reyansh Jain,Kolkata,East,New,Coffee Maker,Home & Kitchen,3499,HomeBrew


Merged shape: (120, 15)


## 3. DateTime operations

The order date is converted to Pandas datetime format. Year, month, month name, day, day name, and year-month are then extracted.

In [6]:
processed['Order_Date'] = pd.to_datetime(processed['Order_Date'])

processed['Order_Year'] = processed['Order_Date'].dt.year
processed['Order_Month'] = processed['Order_Date'].dt.month
processed['Order_Month_Name'] = processed['Order_Date'].dt.month_name()
processed['Order_Day'] = processed['Order_Date'].dt.day
processed['Order_Day_Name'] = processed['Order_Date'].dt.day_name()
processed['Order_Year_Month'] = processed['Order_Date'].dt.strftime('%Y-%m')

display(processed[['Order_Date', 'Order_Year', 'Order_Month', 'Order_Month_Name', 'Order_Day', 'Order_Day_Name', 'Order_Year_Month']].head())

,Order_Date,Order_Year,Order_Month,Order_Month_Name,Order_Day,Order_Day_Name,Order_Year_Month
0,2026-02-19,2026,2,February,19,Thursday,2026-02
1,2026-01-25,2026,1,January,25,Sunday,2026-01
2,2026-02-26,2026,2,February,26,Thursday,2026-02
3,2026-03-04,2026,3,March,4,Wednesday,2026-03
4,2026-03-29,2026,3,March,29,Sunday,2026-03


## 4. Using `apply()` to create useful columns

`apply()` is used to calculate the total order amount and create a customer label combining the customer name and membership type.

In [7]:
processed['Total_Amount'] = processed.apply(
    lambda row: row['Quantity'] * row['Unit_Price'],
    axis=1
)

processed['Customer_Label'] = processed.apply(
    lambda row: f"{row['Customer_Name']} ({row['Membership_Type']})",
    axis=1
)

display(processed[['Customer_Name', 'Membership_Type', 'Customer_Label', 'Quantity', 'Unit_Price', 'Total_Amount']].head())

,Customer_Name,Membership_Type,Customer_Label,Quantity,Unit_Price,Total_Amount
0,Harsh Vardhan,Premium,Harsh Vardhan (Premium),2,2499,4998
1,Ishita Gupta,Premium,Ishita Gupta (Premium),2,899,1798
2,Karan Joshi,Regular,Karan Joshi (Regular),1,3299,3299
3,Maryam Khan,Regular,Maryam Khan (Regular),3,999,2997
4,Reyansh Jain,New,Reyansh Jain (New),5,3499,17495


## 5. Clean and meaningful final DataFrame

The columns are arranged logically and the records are sorted by order date and order ID.

In [8]:
columns = [
    'Order_ID', 'Order_Date', 'Order_Year', 'Order_Month', 'Order_Month_Name',
    'Order_Day', 'Order_Day_Name', 'Order_Year_Month',
    'Customer_ID', 'Customer_Name', 'Customer_Label', 'City', 'Region',
    'Membership_Type', 'Product_ID', 'Product_Name', 'Category', 'Brand',
    'Unit_Price', 'Quantity', 'Total_Amount', 'Payment_Method', 'Order_Status'
]

processed = processed[columns].sort_values(
    ['Order_Date', 'Order_ID']
).reset_index(drop=True)

display(processed.head(10))
print('Final shape:', processed.shape)

,Order_ID,Order_Date,Order_Year,Order_Month,Order_Month_Name,Order_Day,Order_Day_Name,Order_Year_Month,Customer_ID,Customer_Name,...,Membership_Type,Product_ID,Product_Name,Category,Brand,Unit_Price,Quantity,Total_Amount,Payment_Method,Order_Status
0,O0051,2026-01-01,2026,1,January,1,Thursday,2026-01,C008,Sara Ahmed,...,Premium,P017,Yoga Mat,Sports,FitLife,899,4,3596,Net Banking,Delivered
1,O0091,2026-01-01,2026,1,January,1,Thursday,2026-01,C002,Zoya Khan,...,Regular,P011,Air Fryer,Home & Kitchen,CookSmart,4999,2,9998,Credit Card,Delivered
2,O0022,2026-01-02,2026,1,January,2,Friday,2026-01,C023,Nikhil Sood,...,Premium,P014,Data Science Handbook,Books,DataPress,899,4,3596,Net Banking,Delivered
3,O0106,2026-01-02,2026,1,January,2,Friday,2026-01,C001,Aarav Sharma,...,Premium,P009,Coffee Maker,Home & Kitchen,HomeBrew,3499,1,3499,Net Banking,Delivered
4,O0076,2026-01-03,2026,1,January,3,Saturday,2026-01,C019,Yusuf Dar,...,New,P015,Machine Learning Basics,Books,AIPress,999,1,999,Debit Card,Delivered
5,O0041,2026-01-04,2026,1,January,4,Sunday,2026-01,C008,Sara Ahmed,...,Premium,P012,Water Bottle,Home & Kitchen,HydroLife,699,5,3495,Cash on Delivery,Delivered
6,O0108,2026-01-04,2026,1,January,4,Sunday,2026-01,C006,Ishita Gupta,...,Premium,P020,Dumbbell Set,Sports,StrongFit,1999,1,1999,Net Banking,Delivered
7,O0018,2026-01-05,2026,1,January,5,Monday,2026-01,C003,Rohan Mehta,...,Premium,P011,Air Fryer,Home & Kitchen,CookSmart,4999,4,19996,Net Banking,Cancelled
8,O0104,2026-01-05,2026,1,January,5,Monday,2026-01,C011,Vivaan Kapoor,...,New,P006,Hoodie,Clothing,UrbanWear,1599,3,4797,Debit Card,Shipped
9,O0110,2026-01-05,2026,1,January,5,Monday,2026-01,C007,Aditya Verma,...,Regular,P005,Power Bank,Electronics,VoltPlus,1199,1,1199,UPI,Delivered


Final shape: (120, 23)


In [9]:
print('Missing values by column:')
print(processed.isna().sum())

print('\nTotal sales amount:', processed['Total_Amount'].sum())
print('Number of orders:', processed['Order_ID'].nunique())
print('Number of customers:', processed['Customer_ID'].nunique())
print('Number of products:', processed['Product_ID'].nunique())

Missing values by column:
Order_ID            0
Order_Date          0
Order_Year          0
Order_Month         0
Order_Month_Name    0
Order_Day           0
Order_Day_Name      0
Order_Year_Month    0
Customer_ID         0
Customer_Name       0
Customer_Label      0
City                0
Region              0
Membership_Type     0
Product_ID          0
Product_Name        0
Category            0
Brand               0
Unit_Price          0
Quantity            0
Total_Amount        0
Payment_Method      0
Order_Status        0
dtype: int64

Total sales amount: 741507
Number of orders: 120
Number of customers: 30
Number of products: 20


## 6. Export the processed dataset

The final DataFrame is exported as `Day9_Processed_Ecommerce_Dataset.csv`.

In [10]:
output_file = 'Day9_Processed_Ecommerce_Dataset.csv'
processed.to_csv(output_file, index=False)
print(f'Processed dataset exported successfully: {output_file}')

Processed dataset exported successfully: Day9_Processed_Ecommerce_Dataset.csv
